# PD Model

Predicts the probability of default (PD) for each applicant from the feature table built in `06_feature_engineering`.

- **Model:** logistic regression, the standard for credit scorecards because each feature's effect can be explained.
- **Evaluation:** AUC and Gini on a held-out test set (accuracy is misleading with an 8% default rate).
- **Input:** `data/processed/model_features.parquet` (307,511 applicants).

## Setup and load the feature table

In [1]:
%pip install scikit-learn pyarrow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: C:\Users\karol\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

df = pd.read_parquet('data/processed/model_features.parquet')

print(df.shape)
print('Default rate:', round(df['TARGET'].mean() * 100, 2), '%')

(307511, 38)
Default rate: 8.07 %


## Train / test split

80% of applicants train the model; 20% are held back to evaluate it on applicants it has never seen. The split is **stratified** so both sets keep the 8.07% default rate, and fixed with a random seed so results are reproducible.

In [3]:
from sklearn.model_selection import train_test_split

categorical = ['NAME_CONTRACT_TYPE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE']
numeric = [c for c in df.columns if c not in categorical + ['SK_ID_CURR', 'TARGET']]

X = df[categorical + numeric]
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('Train:', X_train.shape, 'default rate', round(y_train.mean() * 100, 2), '%')
print('Test: ', X_test.shape,  'default rate', round(y_test.mean() * 100, 2), '%')
print('Numeric features:', len(numeric))

Train: (246008, 36) default rate 8.07 %
Test:  (61503, 36) default rate 8.07 %
Numeric features: 33


## Capping extreme values

Numeric features are capped at their 1st and 99th percentiles (winsorizing) so that a few extreme values, such as a maximum lateness of 2,884 days, do not dominate the model. Cut-offs are calculated on the training set only and then applied to the test set, to avoid leaking test information into training.

In [10]:
lower = X_train[numeric].quantile(0.01)
upper = X_train[numeric].quantile(0.99)

X_train = X_train.copy()
X_test  = X_test.copy()
X_train[numeric] = X_train[numeric].clip(lower=lower, upper=upper, axis=1)
X_test[numeric]  = X_test[numeric].clip(lower=lower, upper=upper, axis=1)

print('Max days late before capping: 2884')
print('Max days late after capping: ', X_train['pay_max_days_late'].max())

Max days late before capping: 2884
Max days late after capping:  358.0


## Model pipeline

Logistic regression needs complete, numeric, similarly scaled inputs, so preparation and model are combined in one pipeline:

| Step | Numeric features | Text features |
|---|---|---|
| Missing values | Median, plus a 0/1 "was missing" indicator | Most frequent value |
| Transformation | Standardised (mean 0, standard deviation 1) | One-hot encoded (one 0/1 column per category) |

The missing indicators matter because many blanks carry meaning (no bureau record, no payment history). All preparation steps are fitted on the training set only.

In [36]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

numeric_prep = Pipeline([
    ('impute', SimpleImputer(strategy='median', add_indicator=True)),
    ('scale',  StandardScaler()),
])

categorical_prep = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore')),
])

prep = ColumnTransformer([
    ('num', numeric_prep, numeric),
    ('cat', categorical_prep, categorical),
])

model = Pipeline([
    ('prep', prep),
    ('logreg', LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)
print('Model trained')

Model trained


In [37]:
from sklearn.metrics import roc_auc_score

pd_train = model.predict_proba(X_train)[:, 1]
pd_test  = model.predict_proba(X_test)[:, 1]

auc_train = roc_auc_score(y_train, pd_train)
auc_test  = roc_auc_score(y_test,  pd_test)

print(f'Train AUC: {auc_train:.4f}   Gini: {2 * auc_train - 1:.4f}')
print(f'Test  AUC: {auc_test:.4f}   Gini: {2 * auc_test - 1:.4f}')

Train AUC: 0.7508   Gini: 0.5016
Test  AUC: 0.7478   Gini: 0.4956


## Evaluation

| Set | AUC | Gini |
|---|---|---|
| Train | 0.7508 | 0.5016 |
| Test | 0.7478 | 0.4956 |

- **Test Gini of 0.50:** within the range typically seen for retail credit models (about 0.5 to 0.7).
- **No overfitting:** train and test performance are almost identical, so the model generalises to unseen applicants.
- In plain terms: for a random defaulter and a random non-defaulter, the model assigns the defaulter the higher PD about 75% of the time.

In [38]:
from sklearn.metrics import roc_auc_score

ext2_test = X_test['EXT_SOURCE_2'].fillna(X_train['EXT_SOURCE_2'].median())
auc_ext2 = roc_auc_score(y_test, -ext2_test)

print(f'EXT_SOURCE_2 alone  AUC: {auc_ext2:.4f}   Gini: {2 * auc_ext2 - 1:.4f}')
print(f'Full model          AUC: {auc_test:.4f}   Gini: {2 * auc_test - 1:.4f}')

EXT_SOURCE_2 alone  AUC: 0.6541   Gini: 0.3082
Full model          AUC: 0.7478   Gini: 0.4956


## Benchmark: model vs best single feature

| | AUC | Gini |
|---|---|---|
| `EXT_SOURCE_2` alone | 0.6541 | 0.3082 |
| Full model | 0.7478 | 0.4956 |

The full model improves Gini by 0.19 (about 60%) over the strongest single feature found in exploration. The engineered features from bureau, payment and application history add substantial information beyond the external score.

In [40]:
results = pd.DataFrame({'pd': pd_test, 'actual': y_test.values})
results['decile'] = pd.qcut(results['pd'], 10, labels=range(1, 11))

decile_table = results.groupby('decile', observed=True).agg(
    applicants  = ('actual', 'size'),
    avg_pd      = ('pd', 'mean'),
    actual_rate = ('actual', 'mean'),
)
decile_table['avg_pd'] = (decile_table['avg_pd'] * 100).round(2)
decile_table['actual_rate'] = (decile_table['actual_rate'] * 100).round(2)
decile_table

,applicants,avg_pd,actual_rate
decile,,,
1,6151,1.25,1.48
2,6150,2.19,2.26
3,6150,3.02,3.20
4,6150,3.92,3.76
5,6151,4.96,4.47
6,6150,6.27,6.16
7,6150,7.97,7.54
8,6150,10.42,10.94
9,6150,14.50,14.88


## Ranking and calibration by PD decile (test set)

| PD decile | Applicants | Average predicted PD | Actual default rate |
|---|---|---|---|
| 1 (lowest risk) | 6,151 | 1.25% | 1.48% |
| 2 | 6,150 | 2.19% | 2.26% |
| 3 | 6,150 | 3.02% | 3.20% |
| 4 | 6,150 | 3.92% | 3.76% |
| 5 | 6,151 | 4.96% | 4.47% |
| 6 | 6,150 | 6.27% | 6.16% |
| 7 | 6,150 | 7.97% | 7.54% |
| 8 | 6,150 | 10.42% | 10.94% |
| 9 | 6,150 | 14.50% | 14.88% |
| 10 (highest risk) | 6,151 | 26.23% | 26.03% |

- **Strong ranking:** the actual default rate rises in every decile, from 1.48% to 26.03% (a 17 times spread, versus 6 times for `EXT_SOURCE_2` alone).
- **Well calibrated:** predicted and actual default rates differ by at most about half a percentage point, so the PDs can be used directly as probabilities, for example in Expected Loss.
- **Business impact:** the riskiest 10% of applicants contain about 32% of all defaulters in the test set, and the riskiest 20% about half.

In [43]:
feature_names = model.named_steps['prep'].get_feature_names_out()
coefs = model.named_steps['logreg'].coef_[0]

importance = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
importance['abs'] = importance['coefficient'].abs()
importance = importance.sort_values('abs', ascending=False).head(15)
importance[['feature', 'coefficient']].round(3)

,feature,coefficient
57,cat__NAME_CONTRACT_TYPE_Revolving loans,-0.939
67,cat__NAME_EDUCATION_TYPE_Higher education,-0.631
56,cat__NAME_CONTRACT_TYPE_Cash loans,-0.578
62,cat__NAME_INCOME_TYPE_State servant,-0.564
68,cat__NAME_EDUCATION_TYPE_Incomplete higher,-0.494
12,num__EXT_SOURCE_3,-0.420
11,num__EXT_SOURCE_2,-0.411
59,cat__NAME_INCOME_TYPE_Commercial associate,-0.397
61,cat__NAME_INCOME_TYPE_Pensioner,-0.315
70,cat__NAME_EDUCATION_TYPE_Secondary / secondary...,-0.256


In [42]:
feature_names = model.named_steps['prep'].get_feature_names_out()
coefs = model.named_steps['logreg'].coef_[0]

importance = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
importance = importance[importance['feature'].str.startswith('num__')]
importance['abs'] = importance['coefficient'].abs()
importance.sort_values('abs', ascending=False).head(15)[['feature', 'coefficient']].round(3)

,feature,coefficient
12,num__EXT_SOURCE_3,-0.420
11,num__EXT_SOURCE_2,-0.411
10,num__EXT_SOURCE_1,-0.235
17,num__bureau_num_active,0.195
8,num__years_employed,-0.169
29,num__prev_num_approved,-0.162
25,num__pay_share_late,0.155
16,num__bureau_num_loans,-0.147
36,num__missingindicator_EXT_SOURCE_1,0.138
32,num__prev_avg_granted_vs_asked,0.124


## What drives the model

Strongest numeric features (standardised coefficients; positive = higher PD):

| Feature | Coefficient | Reading |
|---|---|---|
| `EXT_SOURCE_3` | -0.420 | Higher external score, lower risk |
| `EXT_SOURCE_2` | -0.411 | Higher external score, lower risk |
| `EXT_SOURCE_1` | -0.235 | Higher external score, lower risk |
| `bureau_num_active` | +0.195 | More active loans elsewhere, higher risk |
| `years_employed` | -0.169 | Longer employment, lower risk |
| `prev_num_approved` | -0.162 | More past approvals, lower risk |
| `pay_share_late` | +0.155 | Larger share of late instalments, higher risk |
| `bureau_num_loans` | -0.147 | Longer loan history (given active loans), lower risk |
| `EXT_SOURCE_1` missing | +0.138 | No score available, higher risk |
| `prev_refusal_rate` | +0.088 | More refusals, higher risk |
| `annuity_to_income` | +0.084 | Heavier repayment burden, higher risk |

- The model's drivers match the exploration findings: external scores, repayment behaviour, refusals and existing debt.
- **Missing-value indicators matter:** a missing `EXT_SOURCE_1` is itself a risk signal.
- **Categorical features** are read within their group: revolving loans are lower risk than cash loans, and risk falls with higher education.
- Several features are correlated, so coefficients indicate which signals matter rather than precise individual effects.

In [44]:
X_all = X.copy()
X_all[numeric] = X_all[numeric].clip(lower=lower, upper=upper, axis=1)

scored = df[['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'AMT_CREDIT', 'age_years',
             'has_bureau_record', 'has_payment_history', 'prev_num_refused']].copy()
scored['pd']  = model.predict_proba(X_all)[:, 1]
scored['set'] = np.where(df.index.isin(X_test.index), 'test', 'train')

LGD = 0.60
scored['ead'] = scored['AMT_CREDIT']
scored['lgd'] = LGD
scored['expected_loss'] = scored['pd'] * scored['ead'] * scored['lgd']

scored['risk_grade'] = pd.qcut(scored['pd'], 5, labels=['A', 'B', 'C', 'D', 'E'])

scored.to_csv('data/processed/scored_applications.csv', index=False)

print(scored.groupby('risk_grade', observed=True).agg(
    applicants = ('pd', 'size'),
    avg_pd     = ('pd', 'mean'),
    actual     = ('TARGET', 'mean'),
    total_el   = ('expected_loss', 'sum'),
).round(4))

            applicants  avg_pd  actual      total_el
risk_grade                                          
A                61503  0.0171  0.0174  4.141661e+08
B                61502  0.0347  0.0341  8.084138e+08
C                61502  0.0562  0.0541  1.251019e+09
D                61502  0.0917  0.0929  1.939646e+09
E                61502  0.2039  0.2051  3.883117e+09


## Expected Loss by risk grade

Every applicant is scored with the model and assigned to one of five equal-sized risk grades (A = lowest PD). Expected Loss uses the formula **EL = PD × EAD × LGD**.

| Grade | Applicants | Average PD | Actual default rate | Total Expected Loss | Share of EL |
|---|---|---|---|---|---|
| A | 61,503 | 1.71% | 1.74% | 414 M | 5% |
| B | 61,502 | 3.47% | 3.41% | 808 M | 10% |
| C | 61,502 | 5.62% | 5.41% | 1,251 M | 15% |
| D | 61,502 | 9.17% | 9.29% | 1,940 M | 23% |
| E | 61,502 | 20.39% | 20.51% | 3,883 M | 47% |

- **Predicted and actual default rates agree** within about 0.2 percentage points in every grade.
- **Expected loss is concentrated:** grade E (20% of applicants) carries 47% of total expected loss, about 9 times that of grade A.
- A lender could address nearly half of expected loss by repricing or limiting exposure in grade E alone, rather than declining applicants broadly.

**Assumptions:** this dataset contains no recovery information, so **LGD is assumed at 60%**, a typical level for unsecured consumer lending. **EAD is approximated by `AMT_CREDIT`**, which overstates exposure for revolving loans (the credit limit, not the drawn balance). Amounts are in an undisclosed currency. Grades are shown for the full portfolio; model performance was validated on the held-out test set.

Output: `data/processed/scored_applications.csv` for the Power BI dashboard.